# Проект спринта 7: тенденции игровой индустрии в 2000-2013 годах

- Автор:Сажина Наталья
- Дата: 19.03.2025

### Описание проекта

Разработчик игры "Секреты Темнолесья" планирует выпустить статью на IT-ресурсе с исследованием игровой индустрии в 2000-2013 годах. Статья должна привлечь внимание людей, которые любят старые игры, и заодно заинтересовать их "Секретами Темнолесья". В рамках проекта будем готовить данные для статьи на основе полученного от заказчика датасета.

### Цели и задачи проекта

Цель проекта - собать данные для информативной статьи о тендециях игровой индустрии в 2000-2013 годах.
    
**Задачи:**

1. Обработать данные, изучить и учесть имеющиеся пропуски и дубликаты, чтобы результаты отражали реальную ситуацию и не были искажены.
    
2. Проанализировать полученные данные и выявить основные тенденции по категориям игр в разрезе оценок пользователей и экспертов и в разрезе разработчиков.


### Описание данных

В проекте будут использованы данные датасета `new_games.csv`, содержщие информацию о продажах игр разных жанров и платформ, а также пользовательские и экспертные оценки игр. Поля датасета:

- `Name` — название игры.

- `Platform` — название платформы.

- `Year of Release` — год выпуска игры.

- `Genre` — жанр игры.

- `NA sales` — продажи в Северной Америке (в миллионах проданных копий).

- `EU sales` — продажи в Европе (в миллионах проданных копий).

- `JP sales` — продажи в Японии (в миллионах проданных копий).

- `Other sales` — продажи в других странах (в миллионах проданных копий).

- `Critic Score` — оценка критиков (от 0 до 100).

- `User Score` — оценка пользователей (от 0 до 10).

- `Rating` — рейтинг организации ESRB (англ. Entertainment Software Rating Board). Эта ассоциация определяет рейтинг компьютерных игр и присваивает им подходящую возрастную категорию.

### Содержимое проекта

1. **Загрузка и знакомство с данными**

2. **Предобработка данных**

3. **Фильтрация данных**

4. **Категоризация данных**

5. **Итоги**

## Загрузка данных и знакомство с ними

- Загрузим `pandas` и данные датасета `/datasets/new_games.csv`.


In [1]:
import pandas as pd

In [2]:
df_source = pd.read_csv('https://code.s3.yandex.net/datasets/new_games.csv')

In [3]:
# При необходимости добавьте новые ячейки для кода

- Познакомимся с данными: выведем первые строки и результат метода `info()`.


In [4]:
df_source.head(6)

,Name,Platform,Year of Release,Genre,NA sales,EU sales,JP sales,Other sales,Critic Score,User Score,Rating
0,Wii Sports,Wii,2006.0,Sports,41.36,28.96,3.77,8.45,76.0,8,E
1,Super Mario Bros.,NES,1985.0,Platform,29.08,3.58,6.81,0.77,NaN,NaN,NaN
2,Mario Kart Wii,Wii,2008.0,Racing,15.68,12.76,3.79,3.29,82.0,8.3,E
3,Wii Sports Resort,Wii,2009.0,Sports,15.61,10.93,3.28,2.95,80.0,8,E
4,Pokemon Red/Pokemon Blue,GB,1996.0,Role-Playing,11.27,8.89,10.22,1.00,NaN,NaN,NaN
5,Tetris,GB,1989.0,Puzzle,23.20,2.26,4.22,0.58,NaN,NaN,NaN


In [5]:
df_source.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16956 entries, 0 to 16955
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Name             16954 non-null  object 
 1   Platform         16956 non-null  object 
 2   Year of Release  16681 non-null  float64
 3   Genre            16954 non-null  object 
 4   NA sales         16956 non-null  float64
 5   EU sales         16956 non-null  object 
 6   JP sales         16956 non-null  object 
 7   Other sales      16956 non-null  float64
 8   Critic Score     8242 non-null   float64
 9   User Score       10152 non-null  object 
 10  Rating           10085 non-null  object 
dtypes: float64(4), object(7)
memory usage: 1.4+ MB


Датасет содержит 11 столбцов и 16956 строк, в которых представлена информация о продажах игр.

Изучим данные и их корректность:

**Заголовки столбцов** содержат пробелы и верхний регистр, что усложнит их упоминание в коде, необходимо привести к snake case.

**Столбцов типа `object`** всего 7: `Name` - название игры, `Platform` - название платформы, `Genre` - жанр игры, `EU sales`, `JP sales`, `User Score`, `Rating`. Для `Name` тип выбран корректно. `Platform`, `Genre` и `Rating` можно рассматривать как категориальные признаки, в этом случае можно использовать тип `category`, чтобы улучшить производительность. Для `EU sales`, `JP sales`, `User Score` необходимо заменить тип на `float64`, т.к. они описывают продажи и рейтинги, представленные дробными значениями.

**Тип `float64`** присвоен остальным столбцам: `Year of Release`, `NA sales`,`Other sales`, `Critic Score`. Для `Year of Release` следует заменить тип данных на целочисленные значения `int16` - битности 16 будет достаточно, т.к. столбец не может содержать значений больше 2025. Остальные столбцы корректно относятся к типу `float64`.

**Пропуски** есть в шести столбцах из 11, часть столбцов с пропусками имеет важное значение для анализа, поэтому пропуски требуют более детольной обработки.

---

## Проверка ошибок в данных и их предобработка


### Названия, или метки, столбцов датафрейма

- Выведем на экран названия всех столбцов датафрейма и проверим их стиль написания.
- Приведем все столбцы к стилю snake case.

In [6]:
df_source.columns

Index(['Name', 'Platform', 'Year of Release', 'Genre', 'NA sales', 'EU sales',
       'JP sales', 'Other sales', 'Critic Score', 'User Score', 'Rating'],
      dtype='object')

In [7]:
#приведем названия столбцов к snake case для удобства использования в коде
df_source.columns = df_source.columns.str.strip().str.lower().str.replace(' ','_')
df_source.head()

,name,platform,year_of_release,genre,na_sales,eu_sales,jp_sales,other_sales,critic_score,user_score,rating
0,Wii Sports,Wii,2006.0,Sports,41.36,28.96,3.77,8.45,76.0,8,E
1,Super Mario Bros.,NES,1985.0,Platform,29.08,3.58,6.81,0.77,NaN,NaN,NaN
2,Mario Kart Wii,Wii,2008.0,Racing,15.68,12.76,3.79,3.29,82.0,8.3,E
3,Wii Sports Resort,Wii,2009.0,Sports,15.61,10.93,3.28,2.95,80.0,8,E
4,Pokemon Red/Pokemon Blue,GB,1996.0,Role-Playing,11.27,8.89,10.22,1.00,NaN,NaN,NaN


### Типы данных

В датасете для ряда столбцов с числовыми данными выбран тип `object`, это может быть связано с наличием пропусков или их строковых индикаторов.

Для столбцов `eu_sales`, `jp_sales`, `user_score` необходимо заменить тип с `object` на `float64`. Для столбца `year_of_release` нужно изменить тип данных с `float64` на `int16`

In [8]:
df_source['eu_sales'] = pd.to_numeric(df_source['eu_sales'], errors = 'coerce')
df_source.info() 

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16956 entries, 0 to 16955
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   name             16954 non-null  object 
 1   platform         16956 non-null  object 
 2   year_of_release  16681 non-null  float64
 3   genre            16954 non-null  object 
 4   na_sales         16956 non-null  float64
 5   eu_sales         16950 non-null  float64
 6   jp_sales         16956 non-null  object 
 7   other_sales      16956 non-null  float64
 8   critic_score     8242 non-null   float64
 9   user_score       10152 non-null  object 
 10  rating           10085 non-null  object 
dtypes: float64(5), object(6)
memory usage: 1.4+ MB


In [9]:
col_to_change = ['eu_sales','jp_sales','user_score'] #задаем список столбцов для приведения к типу float64

for column in col_to_change:
    df_source[column] = pd.to_numeric(df_source[column],errors = 'coerce') #с помощью параметра errors обработаны пропуски и их значения теперь NaN
    
df_source.info()    

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16956 entries, 0 to 16955
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   name             16954 non-null  object 
 1   platform         16956 non-null  object 
 2   year_of_release  16681 non-null  float64
 3   genre            16954 non-null  object 
 4   na_sales         16956 non-null  float64
 5   eu_sales         16950 non-null  float64
 6   jp_sales         16952 non-null  float64
 7   other_sales      16956 non-null  float64
 8   critic_score     8242 non-null   float64
 9   user_score       7688 non-null   float64
 10  rating           10085 non-null  object 
dtypes: float64(7), object(4)
memory usage: 1.4+ MB


In [10]:
df_source['year_of_release'] = df_source['year_of_release'].fillna(0) #меняем пропуски на 0, такого значения не может быть в столбце с годом, он послужит индикатором пропуска
df_source['year_of_release'] = df_source['year_of_release'].astype('int64') #приводим к целочисленному типу
df_source['year_of_release'] = pd.to_numeric(df_source['year_of_release'],downcast = 'integer') #понижаем битность

df_source.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16956 entries, 0 to 16955
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   name             16954 non-null  object 
 1   platform         16956 non-null  object 
 2   year_of_release  16956 non-null  int16  
 3   genre            16954 non-null  object 
 4   na_sales         16956 non-null  float64
 5   eu_sales         16950 non-null  float64
 6   jp_sales         16952 non-null  float64
 7   other_sales      16956 non-null  float64
 8   critic_score     8242 non-null   float64
 9   user_score       7688 non-null   float64
 10  rating           10085 non-null  object 
dtypes: float64(6), int16(1), object(4)
memory usage: 1.3+ MB


Теперь типы данных столбцов соответствуют значениям

### Наличие пропусков в данных

- Посчитаем количество пропусков в каждом столбце в абсолютных и относительных значениях.


In [11]:
df_source.isna().sum() #абсолютное количество пропусков по столбцам

name                  2
platform              0
year_of_release       0
genre                 2
na_sales              0
eu_sales              6
jp_sales              4
other_sales           0
critic_score       8714
user_score         9268
rating             6871
dtype: int64

In [12]:
df_source.isna().mean() #получим долю пропусков для каждого столбца датафрейма 

name               0.000118
platform           0.000000
year_of_release    0.000000
genre              0.000118
na_sales           0.000000
eu_sales           0.000354
jp_sales           0.000236
other_sales        0.000000
critic_score       0.513918
user_score         0.546591
rating             0.405225
dtype: float64

Столбец Name имеет всего два пропуска, при этом в датасете он является **primary key** поэтому строки без этого значения не могут быть обработаны. Необходимо проверить, являются ли эти две строки дубликатами по остальным значениям. Если да - необходимо удалить их. Столько же пропусков видим с столбце Genre, нужно проверить, возможно это те же строки.

Столбец Year of Release содержит около 1,6% пропусков (в предыдущем разделе они заменены на 0), это немного, но для нашего анализ год выхода важный показатель.

Значительную долю пропусков 40-50% имеют столбцы с рейтингами и оценками. Необходимо посмотреть, как изменится доля пропусков при фильтрации датасета по периоду 2000-2023г.

In [13]:
df_actual = df_source[(df_source['year_of_release'] >= 2000) & (df_source['year_of_release'] <= 2013)]
df_actual.info() #узнаем, влияют ли пропуски на нужный нам период 2000-2013

<class 'pandas.core.frame.DataFrame'>
Int64Index: 12980 entries, 0 to 16954
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   name             12980 non-null  object 
 1   platform         12980 non-null  object 
 2   year_of_release  12980 non-null  int16  
 3   genre            12980 non-null  object 
 4   na_sales         12980 non-null  float64
 5   eu_sales         12975 non-null  float64
 6   jp_sales         12976 non-null  float64
 7   other_sales      12980 non-null  float64
 8   critic_score     7267 non-null   float64
 9   user_score       6572 non-null   float64
 10  rating           8847 non-null   object 
dtypes: float64(6), int16(1), object(4)
memory usage: 1.1+ MB


In [14]:
df_actual.isna().sum()

name                  0
platform              0
year_of_release       0
genre                 0
na_sales              0
eu_sales              5
jp_sales              4
other_sales           0
critic_score       5713
user_score         6408
rating             4133
dtype: int64

In [15]:
df_actual.isna().mean()

name               0.000000
platform           0.000000
year_of_release    0.000000
genre              0.000000
na_sales           0.000000
eu_sales           0.000385
jp_sales           0.000308
other_sales        0.000000
critic_score       0.440139
user_score         0.493683
rating             0.318413
dtype: float64

Пропуски в столбцах **`name`** и **`genre`** не относятся к нужному периоду и их пренебрежительно мало - все 2 строки на весь датасет, игнорируем их ниличие.

In [16]:
df_without_year = df_source[df_source['year_of_release'] == 0] #вычисляем абсолютное количество строк с пропущенным годом
absolute = df_without_year.shape[0]
share = round(df_without_year.shape[0] / len(df_source) * 100, 2)

f'всего в столбце с годом выпуска {absolute} пропусков, что составляет {share}% от общего количества строк'

'всего в столбце с годом выпуска 275 пропусков, что составляет 1.62% от общего количества строк'

Столбец **`year_of_release`**: Информация о годе выпуска игр в открытом доступе и ее должно быть возможным спарсить по названию. Но в рамках этого проекта будем игнорировать наличие этих пропусков, т.к. их относительно мало. К тому же нам не потребуется считать динамику по годам, значит наличие пропусков в столбце с годом выпуска не сможет значительно исказить картину. 

Столбцы **`eu_sales`** и **`jp_sales`**: Есть небольшое количество пропусков в столбцах с данными от продажах игр в Евросоюзе и Японии, их всего 4 и 5 соответственно, заполним эти ячейки средними значениями по платформе и году выпуска:

In [17]:
def eu_sales_by_type (row): #функция заменяет пропущенные значения в столбце eu_sales на средние по платформе для того же года
    if pd.isna(row['eu_sales']):
        group = df_source[(df_source['platform'] == row['platform']) & (df_source['year_of_release'] == row['year_of_release'])]
        return group['eu_sales'].mean()
    else:
        return row['eu_sales']
    
df_no_na = df_source    
    
df_no_na['eu_sales'] = df_no_na.apply(eu_sales_by_type, axis = 1)   
    

In [18]:
def jp_sales_by_type (row): #функция заменяет пропущенные значения в столбце jp_sales на средние по платформе для того же года
    if pd.isna(row['jp_sales']):
        group = df_source[(df_source['platform'] == row['platform']) & (df_source['year_of_release'] == row['year_of_release'])]
        return group['jp_sales'].mean()
    else:
        return row['jp_sales']
    
df_no_na['jp_sales'] = df_no_na.apply(jp_sales_by_type, axis = 1)

df_no_na[['eu_sales','jp_sales']].isna().mean() #убедимся, что пропусков в столбцах с данными о продажах не осталось

eu_sales    0.0
jp_sales    0.0
dtype: float64

In [19]:
# Код ревьюера

df_no_na['jp_sales'] = df_no_na['jp_sales']\
    .fillna(df_no_na.groupby(['platform', 'year_of_release'])['jp_sales'].transform('mean'))

Столбцы **`critic_score`** и **`user_score`** непосредственно задействованы в исследовании (понадобятся для категоризации) и имеют значительное количество пропусков в выборке данных по нужному периоду: 44% и 49% соответственно. Способ сбора и хранения  данных об оценке критиков и пользователей может зависеть от платформы, проверим это:

In [20]:
#группируем по платформе и считаем соотношение по количеству игр среди топ-10 платформ

by_platform = df_no_na.groupby('platform')['name'].count() / len(df_no_na)

by_platform = by_platform.sort_values(ascending = False)

by_platform.head(10)

platform
PS2     0.129099
DS      0.128391
PS3     0.079913
Wii     0.079028
X360    0.075548
PSP     0.072482
PS      0.071656
PC      0.058386
XB      0.049481
GBA     0.049363
Name: name, dtype: float64

In [21]:
#посчитаем, на каких платформах чаще встречаются игры с пропуском в user_score
df_no_userscore = df_no_na[df_no_na['user_score'].isna()]

by_platform_no_userscore = df_no_userscore.groupby('platform')['name'].count() / len(df_no_userscore)

by_platform_no_userscore = by_platform_no_userscore.sort_values(ascending = False)

by_platform_no_userscore.head(10)

platform
DS      0.178571
PS      0.113509
PS2     0.100345
PSP     0.086426
Wii     0.083513
GBA     0.062473
PS3     0.050281
3DS     0.038304
N64     0.034851
X360    0.032585
Name: name, dtype: float64

In [22]:
#посчитаем, на каких платформах чаще встречаются игры с пропуском в critic_score

df_no_criticscore = df_no_na[df_no_na['critic_score'].isna()]

by_platform_no_criticscore = df_no_criticscore.groupby('platform')['name'].count() / len(df_no_criticscore)

by_platform_no_criticscore = by_platform_no_criticscore.sort_values(ascending = False)

by_platform_no_criticscore.head(10)

platform
DS      0.166743
PS      0.116250
PS2     0.100413
PSP     0.087216
Wii     0.085954
PS3     0.060018
GBA     0.045215
3DS     0.041428
X360    0.040165
N64     0.037067
Name: name, dtype: float64

Пропуски в оценке от критиков и пользователей более свойственны платформе DS, но отличия несущественны, поэтому предполагаю, что это пропуски MAR. Заполнить пропуски средними значениями не получится - в этом случае мы повлияем на результаты исследования т.к. в категории по оценкам попадут игры, которые могут иметь другую оценку. Во избежание ошибок при применении агрегирующих функций заменим пропуски в этих столбцах на индикаторы. Категория с самой низкой оценкой предполагает значения от 0, значит в качестве индикатора можем использовать значения -1:

In [23]:
df_no_na_score = df_no_na

df_no_na_score[['critic_score','user_score']] = df_no_na_score[['critic_score','user_score']].fillna(-1)

df_no_na_score[['critic_score','user_score']].isna().mean() #убедимся, что в столбцах с оценками не осталось пропусков

critic_score    0.0
user_score      0.0
dtype: float64

Столбец **`rating`** не участвует в анализе и имеет тип данных `object`, значит мы не будем совершать с ним операции, которые могут привести к ошибкам при наличии пропусков. Оставляем в нем пропуски как есть.

После обработки пропусков информация о датафрейме:

In [24]:
df_no_na_score.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16956 entries, 0 to 16955
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   name             16954 non-null  object 
 1   platform         16956 non-null  object 
 2   year_of_release  16956 non-null  int16  
 3   genre            16954 non-null  object 
 4   na_sales         16956 non-null  float64
 5   eu_sales         16956 non-null  float64
 6   jp_sales         16956 non-null  float64
 7   other_sales      16956 non-null  float64
 8   critic_score     16956 non-null  float64
 9   user_score       16956 non-null  float64
 10  rating           10085 non-null  object 
dtypes: float64(6), int16(1), object(4)
memory usage: 1.3+ MB


Пропуски, которые могут помешать рассчетам, заменены на индикаторы.

### Явные и неявные дубликаты в данных

- Изучим уникальные значения в категориальных данных: проверим, встречаются ли среди данных неявные дубликаты, связанные с опечатками или разным способом написания.
- Проведем нормализацию данных с текстовыми значениями.

In [25]:
unique_platform = df_no_na_score['platform'].unique()

ser_platforms = pd.Series(unique_platform) #преобразовала массив в series, чтобы отсортировать результат, так наличие неявных дубликатов будет нагляднее

ser_platforms.sort_values()


15    2600
28     3DO
10     3DS
21      DC
3       DS
2       GB
8      GBA
19      GC
20     GEN
29      GG
11     N64
1      NES
26      NG
14      PC
30    PCFX
12      PS
6      PS2
5      PS3
9      PS4
16     PSP
22     PSV
23     SAT
24     SCD
7     SNES
27    TG16
25      WS
0      Wii
18    WiiU
4     X360
13      XB
17    XOne
dtype: object

Для более наглядных результатов укрупним категории: все версии PS назовем 'PS', то же с 'GBA','PC' и 'Wii':

In [26]:
df_unique_pl = df_no_na_score

df_unique_pl['platform'] = df_unique_pl['platform'].replace({'PS2':'PS','PS3':'PS','PS4':'PS','PSP':'PS','PSV':'PS','WiiU':'Wii','PCFX':'PC','GBA':'GB'})

df_unique_pl['platform'].nunique()

23

In [27]:
unique_genre = df_unique_pl['genre'].unique() #делаем то же самое для жанров

ser_genres = pd.Series(unique_genre) 

ser_genres.sort_values()

16          ACTION
21       ADVENTURE
8           Action
10       Adventure
18        FIGHTING
9         Fighting
13            MISC
5             Misc
20        PLATFORM
23          PUZZLE
1         Platform
4           Puzzle
15          RACING
14    ROLE-PLAYING
2           Racing
3     Role-Playing
17         SHOOTER
22      SIMULATION
19          SPORTS
24        STRATEGY
6          Shooter
7       Simulation
0           Sports
11        Strategy
12             NaN
dtype: object

Среди жанров 25 уникальных значений, множество неявных дубликатов связано с регистром, приведем все к нижнему регистру:

In [28]:
df_unique_genres = df_unique_pl

df_unique_genres['genre'] = df_unique_genres['genre'].str.lower()

unique_lower_genres = df_unique_genres['genre'].unique()

ser_lower_genres = pd.Series(unique_lower_genres)

ser_lower_genres.sort_values()

8           action
10       adventure
9         fighting
5             misc
1         platform
4           puzzle
2           racing
3     role-playing
6          shooter
7       simulation
0           sports
11        strategy
12             NaN
dtype: object

Теперь значения жанров уникальны.

Проверим наличие неявных дубликатов среди значений годов выпуска:

In [29]:
groupedby_year = df_unique_genres.groupby('year_of_release')['name'].count()

groupedby_year

year_of_release
0        275
1980       9
1981      46
1982      37
1983      18
1984      14
1985      14
1986      22
1987      17
1988      15
1989      17
1990      16
1991      42
1992      43
1993      60
1994     121
1995     220
1996     267
1997     293
1998     384
1999     341
2000     357
2001     491
2002     839
2003     789
2004     771
2005     950
2006    1020
2007    1218
2008    1445
2009    1450
2010    1279
2011    1149
2012     670
2013     552
2014     584
2015     612
2016     507
Name: name, dtype: int64

Среди значений года выпуска нет дубликатов.


Уточним наличие неявных дублей в столбце `rating`:

In [30]:
unique_rating = df_unique_genres['rating'].unique()

ser_rating = pd.Series(unique_rating) 

ser_rating.sort_values()

6      AO
0       E
4    E10+
7      EC
5     K-A
2       M
8      RP
3       T
1     NaN
dtype: object

Среди значений `rating` нет неявных дубликатов, обработка неявных дубликатов завершена.


Проверим наличие явных дубликатов:

In [31]:
duplicates = df_unique_genres[df_source.duplicated(keep = False)]

duplicates

,name,platform,year_of_release,genre,na_sales,eu_sales,jp_sales,other_sales,critic_score,user_score,rating
267,Batman: Arkham Asylum,PS,2009,action,2.24,1.31,0.07,0.61,91.0,8.9,T
268,Batman: Arkham Asylum,PS,2009,action,2.24,1.31,0.07,0.61,91.0,8.9,T
367,James Bond 007: Agent Under Fire,PS,2001,shooter,1.90,1.13,0.10,0.41,72.0,7.9,T
368,James Bond 007: Agent Under Fire,PS,2001,shooter,1.90,1.13,0.10,0.41,72.0,7.9,T
716,God of War: Ascension,PS,2013,action,1.23,0.63,0.04,0.35,80.0,7.5,M
...,...,...,...,...,...,...,...,...,...,...,...
16912,Metal Gear Solid V: The Definitive Experience,XOne,2016,action,0.01,0.00,0.00,0.00,-1.0,-1.0,M
16931,Dynasty Warriors: Eiketsuden,PS,2016,action,0.00,0.00,0.01,0.00,-1.0,-1.0,NaN
16939,The Longest 5 Minutes,PS,2016,action,0.00,0.00,0.01,0.00,-1.0,-1.0,NaN
16940,The Longest 5 Minutes,PS,2016,action,0.00,0.00,0.01,0.00,-1.0,-1.0,NaN


520 строк являются дубликатами при том, что значение `name` должно быть уникальным. Удалим явные дубликаты:

In [32]:
df_clear = df_unique_genres.drop_duplicates() #удаляем все повторяющиеся строки кроме первого встреченного значения

df_clear = df_clear.reset_index()

df_clear.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16696 entries, 0 to 16695
Data columns (total 12 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   index            16696 non-null  int64  
 1   name             16694 non-null  object 
 2   platform         16696 non-null  object 
 3   year_of_release  16696 non-null  int16  
 4   genre            16694 non-null  object 
 5   na_sales         16696 non-null  float64
 6   eu_sales         16696 non-null  float64
 7   jp_sales         16696 non-null  float64
 8   other_sales      16696 non-null  float64
 9   critic_score     16696 non-null  float64
 10  user_score       16696 non-null  float64
 11  rating           9948 non-null   object 
dtypes: float64(6), int16(1), int64(1), object(4)
memory usage: 1.4+ MB


In [33]:
df_before_changes = pd.read_csv('https://code.s3.yandex.net/datasets/new_games.csv')

diff = len(df_before_changes) - len(df_clear)

diff_share = round( diff / len(df_before_changes) * 100, 2)

f'Всего в процессе предобработки было удалено {diff} строк, что составляет {diff_share}% от размера изначального датасета. Потери менее 2% можно считать несущественными'

'Всего в процессе предобработки было удалено 260 строк, что составляет 1.53% от размера изначального датасета. Потери менее 2% можно считать несущественными'

После удаления дубликатов осталось 16696 значений из исходных 16956. Удалено 260 полных дубликатов, которые были найдены, после приведения категориальных значений к единому виду.

**Результат предобработки данных**



Были загружены данные `new_games.csv`. Они содержат 11 столбцов и 16956 строк, в которых представлена информация о продажах и рейтингах игр. Названия столбцов датасета приведены к snake case для удобства их использования в коде.

При первичном знакомстве с данными и их предобработке получили такие результаты:

Типы данных изменили для столбцов `eu_sales`,`jp_sales`,`user_score` на `float64`, для столбца `year_of_release` тип данных изменен на `int16`


В столбцах `name`, `genre`, `year_of_release`, `eu_sales`, `jp_sales`, `user_score`, `critic_score`, `rating` были найдены пропуски. 

Для `year_of_release`, `user_score`, `critic_score` пропуски заменены на индикаторы, которые помогут избежать ошибок и не исказят результаты исследования. 

В столбцах `eu_sales`, `jp_sales` пропуски заменены на средние значения для платформы и года.

В столбцах `name`, `genre` и `rating` пропуски игнорируем, они не помешают обработке данных.


В столбцах `genre` и `platform` неявные дубликаты приведены к единому виду, что позволило следующим шагом найти и удалить 260 явных дубликатов.

Датафрейм **`df_clear`** готов к анализу


---

## Фильтрация данных

Согласно задачам исследования отберем данные по периоду 2000-2013 год включетельно

In [34]:
df_actual = df_clear[(df_clear['year_of_release'] >= 2000) & (df_clear['year_of_release'] <= 2013)]

df_actual = df_actual.reset_index()

df_actual = df_actual.drop('index', axis = 1) #удаляем лишние столбцы, которые образовались из-за подмены индексов при группировке и фильтрации 
df_actual = df_actual.drop('level_0', axis = 1)

df_actual.info()                     

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12773 entries, 0 to 12772
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   name             12773 non-null  object 
 1   platform         12773 non-null  object 
 2   year_of_release  12773 non-null  int16  
 3   genre            12773 non-null  object 
 4   na_sales         12773 non-null  float64
 5   eu_sales         12773 non-null  float64
 6   jp_sales         12773 non-null  float64
 7   other_sales      12773 non-null  float64
 8   critic_score     12773 non-null  float64
 9   user_score       12773 non-null  float64
 10  rating           8723 non-null   object 
dtypes: float64(6), int16(1), object(4)
memory usage: 1023.0+ KB


В выборке данных по нужному периоду 2000-2013 год 12773 строки, пропуски содержит только столбец с рейтингом.

---

## Категоризация данных
    
Проведем категоризацию данных:
- Разделим все игры по оценкам пользователей и выделим такие категории: высокая оценка (от 8 до 10 включительно), средняя оценка (от 3 до 8, не включая правую границу интервала) и низкая оценка (от 0 до 3, не включая правую границу интервала).

In [35]:
df_with_cat = df_actual

def categorize_byusers (row):
    if row['user_score'] >= 0 and row['user_score'] < 3:
        return 'Низкая оценка'
    if row['user_score'] >= 3 and row['user_score'] < 8:
        return 'Средняя оценка'
    if row['user_score'] >= 8 and row['user_score'] <= 10:
        return 'Высокая оценка'
    else:
        return 'Нет данных'

df_with_cat['category_user_score'] = df_with_cat.apply(categorize_byusers, axis = 1)
df_with_cat.head(7)

,name,platform,year_of_release,genre,na_sales,eu_sales,jp_sales,other_sales,critic_score,user_score,rating,category_user_score
0,Wii Sports,Wii,2006,sports,41.36,28.96,3.77,8.45,76.0,8.0,E,Высокая оценка
1,Mario Kart Wii,Wii,2008,racing,15.68,12.76,3.79,3.29,82.0,8.3,E,Высокая оценка
2,Wii Sports Resort,Wii,2009,sports,15.61,10.93,3.28,2.95,80.0,8.0,E,Высокая оценка
3,New Super Mario Bros.,DS,2006,platform,11.28,9.14,6.50,2.88,89.0,8.5,E,Высокая оценка
4,Wii Play,Wii,2006,misc,13.96,9.18,2.93,2.84,58.0,6.6,E,Средняя оценка
5,New Super Mario Bros. Wii,Wii,2009,platform,14.44,6.94,4.70,2.24,87.0,8.4,E,Высокая оценка
6,Nintendogs,DS,2005,simulation,9.05,10.95,1.93,2.74,-1.0,-1.0,NaN,Нет данных


- Разделим все игры по оценкам критиков и выделим такие категории: высокая оценка (от 80 до 100 включительно), средняя оценка (от 30 до 80, не включая правую границу интервала) и низкая оценка (от 0 до 30, не включая правую границу интервала).

In [36]:
df_with_two_cats = df_with_cat

def categorize_bycritics (row):
    if row['critic_score'] >= 0 and row['critic_score'] < 30:
        return 'Низкая оценка'
    if row['critic_score'] >= 30 and row['critic_score'] < 80:
        return 'Средняя оценка'
    if row['critic_score'] >= 80 and row['critic_score'] <= 100:
        return 'Высокая оценка'
    else:
        return 'Нет данных'



df_with_two_cats['category_critic_score'] = df_with_two_cats.apply(categorize_bycritics, axis = 1)

df_with_two_cats.head(7)

,name,platform,year_of_release,genre,na_sales,eu_sales,jp_sales,other_sales,critic_score,user_score,rating,category_user_score,category_critic_score
0,Wii Sports,Wii,2006,sports,41.36,28.96,3.77,8.45,76.0,8.0,E,Высокая оценка,Средняя оценка
1,Mario Kart Wii,Wii,2008,racing,15.68,12.76,3.79,3.29,82.0,8.3,E,Высокая оценка,Высокая оценка
2,Wii Sports Resort,Wii,2009,sports,15.61,10.93,3.28,2.95,80.0,8.0,E,Высокая оценка,Высокая оценка
3,New Super Mario Bros.,DS,2006,platform,11.28,9.14,6.50,2.88,89.0,8.5,E,Высокая оценка,Высокая оценка
4,Wii Play,Wii,2006,misc,13.96,9.18,2.93,2.84,58.0,6.6,E,Средняя оценка,Средняя оценка
5,New Super Mario Bros. Wii,Wii,2009,platform,14.44,6.94,4.70,2.24,87.0,8.4,E,Высокая оценка,Высокая оценка
6,Nintendogs,DS,2005,simulation,9.05,10.95,1.93,2.74,-1.0,-1.0,NaN,Нет данных,Нет данных


- Проверим результат категоризации: сгруппируем данные по выделенным категориям и посчитаем количество игр в каждой категории:

In [37]:
#группировка по категориям оценки пользователей

df_by_userscore = df_with_two_cats.groupby('category_user_score', as_index = False)['name'].count()

df_by_userscore['share_perc'] = df_by_userscore['name'] / df_with_two_cats.shape[0] *100

df_by_userscore

,category_user_score,name,share_perc
0,Высокая оценка,2286,17.897127
1,Нет данных,6290,49.244500
2,Низкая оценка,116,0.908166
3,Средняя оценка,4081,31.950207


In [38]:
df_by_criticscore = df_with_two_cats.groupby('category_critic_score', as_index = False)['name'].count()

df_by_criticscore['share_perc'] = df_by_criticscore['name'] / df_with_two_cats.shape[0] *100

df_by_criticscore

,category_critic_score,name,share_perc
0,Высокая оценка,1692,13.246692
1,Нет данных,5604,43.873796
2,Низкая оценка,55,0.430596
3,Средняя оценка,5422,42.448916


Соотношения размеров категорий по оценкам пользователей и критиков похожи - наибольшую долю занимают игры со средней оценкой. Следующая по размеру категория - игры с высокой оценкой. Игр с низкой оценкой значительно меньше - 0,9% по мнению пользователей и 0,43% по мнению критиков. У значительного количества игр нет даных об оценке - 43,9% для оценки критиков и 49,2% для оценки пользователей. 

- Выделим топ-7 платформ по количеству игр, выпущенных за весь актуальный период.

In [39]:
actual_groupedby_platform = df_with_two_cats.groupby('platform', as_index = False)['name'].count()

actual_groupedby_platform = actual_groupedby_platform.sort_values(by = 'name', ascending = False)

actual_groupedby_platform.head(7)

,platform,name
7,PS,4810
2,DS,2120
9,Wii,1349
10,X360,1121
3,GB,838
11,XB,803
6,PC,766


Первое место по количеству выпущенных игр за 2000-2013 годы с большим отрывом занимает PS.

---

## Итоговый вывод


В рамках подготовлены данные для статьи на основе полученного от заказчика датасета - new_games.csv, содержащего информацию о продажах игр разных жанров и платформ, а также пользовательские и экспертные оценки игр.

Исходный датасет содержит 11 столбцов и 16956 строк, в которых представлена информация о продажах игр.

##### Этапы проекта

1. Загружены данные `new_games.csv`. Они содержат 11 столбцов и 16956 строк, в которых представлена информация о продажах и рейтингах игр. Названия столбцов датасета приведены к snake case для удобства их использования в коде.


2. Типы данных изменили для столбцов `eu_sales`,`jp_sales`,`user_score` на `float64`, для столбца `year_of_release` тип данных изменен на `int16`


3. Обработаны пропуски в данных: 
- В столбцах `name`, `genre`, `year_of_release`, `eu_sales`, `jp_sales`, `user_score`, `critic_score`, `rating` были найдены пропуски. 
- Для `year_of_release`, `user_score`, `critic_score` пропуски заменены на индикаторы, которые помогли избежать ошибок и не исказили результаты исследования.
- В столбцах `eu_sales`, `jp_sales` пропуски заменены на средние значения для платформы и года.
- В столбцах `name`, `genre` и `rating` пропуски игнорируем, они не помешали обработке данных.


4. Проведены поиск и обработке дубликатов: в столбцах `genre` и `platform` неявные дубликаты приведены к единому виду, что позволило следующим шагом найти и удалить 260 явных дубликатов. В результате предобработки данных мы получили датафрейм без дубликатов и критичных пропусков с корректными типами столбцов, в нем 16696 строк и 11 столбцов.


5. Согласно задачам исследования выделили нужную выборку данных с помощью фильтрации по периоду 2000 - 2013 год. После фильтрации данных по периоду мы получили датафрейм с 12773 строками.


6. Для получения результатов исследования были добавлены столбцы с категориями, присвоенными играм по значению оценки пользователей и оценки критиков: `category_user_score` и `category_critic_score`. Для каждой категории подсчитано абсолютное и относительное количество значений. Выяснили, что чаще всего и пользователи и критики дают игре среднюю оценку, следующая по частоте - высокая оценка, причем пользователи склонны ставить высокую оценку чаще, чем критики - 13% у пользователей против 17% у критиков. Крайне редко (менее одного процента случаев) игра получает низкую оценку от критиков или пользователей. При этом необходимо учитывать, что для значительной части игр (43%-49%) неизвестна оценка пользователей или критиков.


7. Далее данные были сгруппированы по игровым платфомам с целью составления топ-7 платформ по количеству игр, выпущенных за рассматриваемый период. Наиболее продуктивной оказалась платформа PS с 4810 выпущенных игр.